# Module 00, Notebook 1: Environment Setup

## Learning Objectives
By completing this notebook, you will:
1. Install CausalPy, PyMC, ArviZ, and all required dependencies
2. Verify your installation by running a complete import check
3. Confirm GPU/CPU backend is configured correctly for PyMC sampling
4. Run a minimal smoke test of the CausalPy API

## Prerequisites
- Python 3.10 or 3.11 (CausalPy requires Python >= 3.10)
- A working conda or virtualenv environment

## Estimated Time: 10 minutes

---

## 1. Installation

Install the required packages. Run this cell once, then restart the kernel.

In [ ]:
# Install all required packages
# Run this cell once, then restart the kernel before proceeding

import subprocess
import sys

packages = [
    "causalpy",
    "pymc>=5.0",
    "arviz>=0.17",
    "pandas>=2.0",
    "numpy>=1.24",
    "matplotlib>=3.7",
    "scipy>=1.11",
    "networkx>=3.0",
    "pytensor>=2.18",
]

for package in packages:
    print(f"Installing {package}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package, "-q"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[:200]}")
    else:
        print(f"  OK")

print("\nInstallation complete. Restart the kernel now.")

## 2. Import Verification

After restarting the kernel, run the following cell to verify all imports succeed and print version numbers.

In [ ]:
# Verify all imports and print version information
# This cell should run without errors after a successful installation

import importlib

required_packages = {
    "causalpy": "causalpy",
    "pymc": "pymc",
    "arviz": "arviz",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "networkx": "networkx",
    "pytensor": "pytensor",
}

all_ok = True
print("Package Version Check")
print("=" * 40)

for display_name, module_name in required_packages.items():
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  {display_name:<15} {version}")
    except ImportError as e:
        print(f"  {display_name:<15} MISSING — {e}")
        all_ok = False

print("=" * 40)
if all_ok:
    print("All packages imported successfully.")
else:
    print("Some packages are missing. Re-run the installation cell.")

## 3. PyMC Sampler Check

CausalPy uses PyMC for Bayesian inference. PyMC can use multiple backends for sampling:
- **CPU (default):** Uses PyTensor with CPU. Works everywhere.
- **JAX backend:** Faster on modern hardware via `numpyro` or `blackjax`.

The following cell checks which backend is active and runs a minimal sampling test to confirm PyMC works end-to-end.

In [ ]:
# Test that PyMC can build and sample a simple model
# This confirms the entire computational stack (pytensor, numba, NUTS sampler) works

import numpy as np
import pymc as pm
import arviz as az

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print()

# Generate simple data: y ~ Normal(mu=5, sigma=2)
np.random.seed(42)
observed_data = np.random.normal(loc=5.0, scale=2.0, size=50)

# Build a minimal Bayesian model to infer the mean
with pm.Model() as sanity_check_model:
    # Prior on the mean: weakly informative
    mu = pm.Normal("mu", mu=0, sigma=10)
    # Prior on standard deviation
    sigma = pm.HalfNormal("sigma", sigma=5)
    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=observed_data)

    # Sample with NUTS — this is the key test
    idata = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        progressbar=True,
        random_seed=42,
    )

# Check the posterior mean estimate
posterior_mu = idata.posterior["mu"].values.flatten()
estimated_mean = posterior_mu.mean()
true_mean = 5.0

print(f"\nSampling completed successfully.")
print(f"True mean: {true_mean:.2f}")
print(f"Estimated posterior mean: {estimated_mean:.2f}")
print(f"95% HDI: [{np.percentile(posterior_mu, 2.5):.2f}, {np.percentile(posterior_mu, 97.5):.2f}]")

if abs(estimated_mean - true_mean) < 0.5:
    print("\nPyMC sanity check PASSED — sampler is working correctly.")
else:
    print("\nWARNING: Estimated mean is far from true mean. Check your installation.")

## 4. CausalPy API Smoke Test

Run a minimal end-to-end test of the CausalPy API. This creates a small synthetic ITS dataset and fits a model to confirm the library works.

In [ ]:
# Minimal CausalPy smoke test
# Creates a synthetic ITS dataset and fits InterruptedTimeSeries

import causalpy as cp
import pandas as pd
import numpy as np

print(f"CausalPy version: {cp.__version__}")
print("Running CausalPy smoke test...")

# Generate a simple ITS dataset
# Pre-intervention: 50 time points with a positive trend
# Post-intervention: 50 time points with the same trend + a +5 level shift
np.random.seed(42)
n_pre = 50
n_post = 50
n_total = n_pre + n_post

t = np.arange(n_total)
treated = (t >= n_pre).astype(int)  # Indicator: 1 after intervention
t_post = np.maximum(t - n_pre, 0)   # Time since intervention (0 in pre-period)

# True data-generating process:
# y = 10 + 0.5*t + 5*treated + 0.1*t_post*treated + noise
true_level_change = 5.0
true_slope_change = 0.1

y = (
    10
    + 0.5 * t
    + true_level_change * treated
    + true_slope_change * t_post * treated
    + np.random.normal(0, 2, n_total)
)

df = pd.DataFrame({
    "t": t,
    "y": y,
    "treated": treated,
    "t_post": t_post,
})

print(f"Dataset shape: {df.shape}")
print(f"Pre-intervention observations: {(df['treated'] == 0).sum()}")
print(f"Post-intervention observations: {(df['treated'] == 1).sum()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nLast few rows:")
print(df.tail())

In [ ]:
# Fit the ITS model using CausalPy
# This tests the full pipeline: model building, sampling, and posterior extraction

# The formula specifies the regression structure:
# t: secular time trend
# treated: level change at intervention
# t_post: slope change after intervention (time since intervention)

its_model = cp.InterruptedTimeSeries(
    data=df,
    treatment_time=n_pre,  # Intervention occurs at t=50
    formula="y ~ 1 + t + treated + t_post",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            "draws": 500,
            "tune": 500,
            "chains": 2,
            "progressbar": True,
            "random_seed": 42,
        }
    ),
)

print("\nCausalPy ITS model fitted successfully.")

In [ ]:
# Visualize the ITS results
# The plot shows: observed data, fitted model, counterfactual, and causal impact

import matplotlib.pyplot as plt

fig, ax = its_model.plot()
ax[0].set_title("CausalPy Smoke Test: Interrupted Time Series", fontsize=13)
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Outcome")
plt.tight_layout()
plt.show()

# Print the summary of causal impact
print("\nCausal Impact Summary:")
print(its_model.summary())

## 5. ArviZ Visualization Check

ArviZ provides diagnostic plots for Bayesian models. Run this cell to confirm ArviZ is working and can render posterior plots.

In [ ]:
# Verify ArviZ can render posterior plots
# InferenceData is the common format shared between PyMC and ArviZ

import arviz as az
import matplotlib.pyplot as plt

# Plot the posterior of the model coefficients
az.plot_posterior(
    its_model.idata,
    var_names=["Intercept", "t", "treated", "t_post"],
    figsize=(12, 4),
)
plt.suptitle("Posterior Distributions of ITS Model Coefficients", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Print R-hat convergence diagnostics (should be close to 1.0)
print("\nConvergence Diagnostics (R-hat, should be < 1.01):")
rhat = az.rhat(its_model.idata)
for var in ["Intercept", "t", "treated", "t_post"]:
    if var in rhat:
        print(f"  {var}: {float(rhat[var].values):.4f}")

## 6. Installation Summary

If all cells above ran without errors, your environment is correctly configured.

In [ ]:
# Final installation summary

import causalpy as cp
import pymc as pm
import arviz as az
import pandas as pd
import numpy as np
import matplotlib
import scipy

print("Environment Setup Summary")
print("=" * 50)
print(f"  Python:      {sys.version.split()[0]}")
print(f"  CausalPy:    {cp.__version__}")
print(f"  PyMC:        {pm.__version__}")
print(f"  ArviZ:       {az.__version__}")
print(f"  pandas:      {pd.__version__}")
print(f"  numpy:       {np.__version__}")
print(f"  matplotlib:  {matplotlib.__version__}")
print(f"  scipy:       {scipy.__version__}")
print("=" * 50)
print("Status: READY")
print()
print("Next notebook: 02_first_causal_analysis.ipynb")
print("  -> Run a complete ITS on real data with CausalPy")

## Summary

### What You Set Up
1. **CausalPy** — high-level API for causal inference models
2. **PyMC** — Bayesian modeling and NUTS sampling
3. **ArviZ** — Bayesian diagnostics and visualization
4. **Supporting libraries** — pandas, numpy, matplotlib, scipy

### The Computational Stack
```
CausalPy (high-level API)
    ↓
PyMC (model specification + sampling)
    ↓
PyTensor (computational graph)
    ↓
NUTS sampler (gradient-based MCMC)
    ↓
ArviZ (diagnostics + visualization)
```

### What's Next
In the next notebook, you will run a real causal analysis using CausalPy's `InterruptedTimeSeries` model on a public policy dataset. You will see the full workflow: loading data, fitting the model, and interpreting the causal impact estimate.